# Notebook 06. 최종 결과와 Human-in-the-Loop 인계

## 분석 질문

현재 어떤 업종을 무엇 때문에 확인해야 하며, 담당자가 무엇을 확인한 뒤 어디로 연결할 수 있는가?

- <b>선행 단계</b> — `05_external_evidence.ipynb`에서 외부근거는 판정을 덮어쓰지 않고 변화의 맥락과 확인질문을 보강한다는 원칙을 확인했다.

## 입력

| 구분 | 경로 |
|---|---|
| Triage 패널 | `outputs/final_model/02_triage/tables/triage_panel.csv` |
| 인계 근거(explanation trace) | `outputs/final_model/05_handoff/tables/explanation_trace.csv` |
| 외부근거 요약 | `outputs/final_model/04_external_evidence/external_evidence_summary.csv` |
| 선택적 재검토 결과 | `outputs/final_model/03_electre_smaa/tables/electre_smaa_review_cases.csv` |
| 인계 기관 지도 | `outputs/final_model/05_handoff/tables/institution_routing_map.csv` |

## 출력

이 노트북은 새 표·그림 파일을 저장하지 않는다. 기존 산출물을 불러와 진단카드와 인계 지도로 정리한다.

## 구성

| 파트 | 내용 | 성격 |
|---|---|---|
| 1 | 산출물 불러오기와 무결성 확인 | 확증 |
| 2 | 최신분기 한눈에 보기 | 확증 |
| 3 | 중요 사례 진단카드 | 확증 |
| 4 | 선택적 ELECTRE/SMAA와 Human-in-the-Loop | 확증 |
| 5 | 기존 기능으로의 인계 | 확증 |
| 6 | 담당자 기록 필드와 다음 분기 재점검 | 확증 |

## 전체 수행 흐름

```text
Q1~Q3 CORE
     ↓
   Triage
     ├─ 관찰 ─────────────────────→ 정기 모니터링
     ├─ 추가확인 → Selective ELECTRE/SMAA → 외부근거 확인
     └─ 우선점검 ─────────────────→ 외부근거 확인
                                      ↓
                             담당자 Human-in-the-Loop
                                      ↓
                              기업·현장 추가 확인
                                      ↓
                 기존 산업동향·기업·고용·훈련·위기대응 기능 인계
                                      ↓
                              다음 분기 재점검
```

ELECTRE는 모든 경로의 필수 단계가 아니며 추가확인에서만 선택적으로 진입한다.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from utils import notebook_config as config
from core.analysis.config import setup_matplotlib
setup_matplotlib()
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)

def _md_value(value):
    if pd.isna(value): return '—'
    if isinstance(value, (float, np.floating)): return f'{value:,.2f}'
    return str(value).replace('|', '\\|').replace('\n', ' / ')

def show_table(frame, max_rows=30):
    view=frame.head(max_rows)
    header='| '+' | '.join(map(str,view.columns))+' |'
    rule='| '+' | '.join(['---']*len(view.columns))+' |'
    rows=['| '+' | '.join(_md_value(v) for v in row)+' |' for row in view.itertuples(index=False,name=None)]
    note=f'\n\n*상위 {max_rows}행만 표시*' if len(frame)>max_rows else ''
    display(Markdown('\n'.join([header,rule,*rows])+note))

def show_records(frame, title_col, fields=None, max_rows=30):
    fields=fields or [c for c in frame.columns if c!=title_col]
    blocks=[]
    for _, row in frame.head(max_rows).iterrows():
        blocks.append('#### '+_md_value(row[title_col]))
        blocks.extend(f'- **{c}:** {_md_value(row[c])}' for c in fields)
    if len(frame)>max_rows: blocks.append(f'*상위 {max_rows}건만 표시*')
    display(Markdown('\n\n'.join(blocks)))

STAGE_ORDER = ['관찰', '추가확인', '우선점검', '자료확인']
STAGE_COLORS = {'관찰':'#7895A8', '추가확인':'#D39A2C', '우선점검':'#C5523F', '자료확인':'#777777'}
STATE_ORDER = ['S1', 'S2', 'S3', 'S4', 'N', 'INVALID']
STATE_COLORS = {'S1':'#4C78A8','S2':'#72B7B2','S3':'#F2CF5B','S4':'#B279A2','N':'#BDBDBD','INVALID':'#F2F2F2'}

## 1단계. 산출물 불러오기와 무결성 확인

### 왜 이걸 보나

진단카드와 인계 지도는 Triage·인계·외부근거·선택적 재검토 산출물을 함께 쓴다. 먼저 최신분기 업종 수와 ELECTRE 진입 규칙이 산출물에서도 지켜지는지 코드로 확인한다.

In [2]:
triage=pd.read_csv(config.TRIAGE_PANEL_PATH,encoding='utf-8-sig'); trace=pd.read_csv(config.HANDOFF_TRACE_PATH,encoding='utf-8-sig')
evidence=pd.read_csv(config.EVIDENCE_SUMMARY_PATH,encoding='utf-8-sig'); review=pd.read_csv(config.ELECTRE_REVIEW_PATH,encoding='utf-8-sig')
routing=pd.read_csv(config.ROUTING_MAP_PATH,encoding='utf-8-sig'); latest_q=triage.quarter.max(); latest=triage.loc[triage.quarter.eq(latest_q)].copy()
assert len(latest)==10 and len(trace)==10
expected=trace.stage.eq('추가확인'); actual=trace.electre_used.astype(str).str.lower().eq('true'); assert actual.equals(expected)
print(f'최신분기 {latest_q} | stage={latest.stage.value_counts().to_dict()} | Human review required={trace.human_decision_required.unique().tolist()}')

최신분기 2026Q2 | stage={'관찰': 8, '우선점검': 2} | Human review required=[True]


### 관찰 결과

- 최신분기 업종은 10개이고 인계 근거(trace)도 10행으로 같다.
- `electre_used`가 True인 행과 `stage`가 추가확인인 행이 정확히 일치한다. 최신분기는 추가확인이 0건이므로 이번 분기는 모두 False다.
- 최신 10건 모두 `human_decision_required = True`다.

> 주의: 이 확인은 산출물 간 정합성 점검이며, 담당자 확인 결과는 아직 반영되지 않은 값이다.

## 2단계. 최신분기 한눈에 보기

결론을 뒤에 숨기지 않고 최신 10개 업종을 먼저 제시한다. `외부근거 가용`은 적어도 하나의 맥락자료가 있다는 뜻이지 판정에 쓸 수 있다는 뜻이 아니다.

In [3]:
summary=trace.merge(evidence[['industry','signal_profile','ppi_mapping_grade','trade_export_yoy','power_usage_yoy']],on=['industry','signal_profile'],how='left',validate='one_to_one')
summary['triggered_signals']=summary.apply(lambda r:'|'.join(x for x,c in [('E',r.signal_e>=5),('R',r.signal_r>=5),('A',r.signal_a>=1),('P',r.signal_p>=5)] if c) or '없음',axis=1)
summary['external_available']=summary[['ppi_mapping_grade','trade_export_yoy','power_usage_yoy']].notna().any(axis=1)
out=summary[['industry','stage','triggered_signals','q2_employment_delta','q2_employment_share','q3_state_run_length','q3_repeated_signal','external_available','signal_profile','stage_reason']].copy()
out.columns=['업종','단계','핵심 trigger','고용증감(명)','고용비중(%)','상태지속(Q)','고용진입 반복','외부근거 가용','외부 맥락','주요 해석/선정 이유']
order={'우선점검':0,'추가확인':1,'관찰':2,'자료확인':3}; out=out.assign(_o=out['단계'].map(order)).sort_values(['_o','고용증감(명)']).drop(columns='_o')
show_table(out.round(2))

| 업종 | 단계 | 핵심 trigger | 고용증감(명) | 고용비중(%) | 상태지속(Q) | 고용진입 반복 | 외부근거 가용 | 외부 맥락 | 주요 해석/선정 이유 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 기계 | 우선점검 | E\|A\|P | -3,979.00 | 51.19 | 2.00 | False | True | 가동 위축 신호 · 실질생산 위축 신호(PPI 조정) | 고용감소 6.3%(진입경계 5% 통과) · 산단 제조업 고용의 3.47% 감소(상위경계 2.0% 통과) → 보강: 생산 19.5% 감소 |
| 목재종이 | 우선점검 | E\|R\|P | -48.00 | 0.37 | 2.00 | True | True | 실질생산 위축 신호(PPI 조정) | 고용감소 10.1%(상위경계 10% 통과) · 산단평균 대비 6.4%p 열위(진입경계 통과) → 보강: 생산 12.9% 감소 / 직전 분기에도 진입신호(지속) |
| 운송장비 | 관찰 | 없음 | -129.00 | 14.20 | 2.00 | False | True | 해석 보류 | 고용 축 진입신호 없음 |
| 전기전자 | 관찰 | 없음 | -116.00 | 24.37 | 1.00 | False | True | 해석 보류 | 고용 축 진입신호 없음 |
| 철강 | 관찰 | 없음 | -112.00 | 8.44 | 4.00 | False | True | 생산유지·고용감소 신호 | 고용 축 진입신호 없음 |
| 음식료 | 관찰 | 없음 | -25.00 | 0.55 | 2.00 | False | True | 해석 보류 | 고용 축 진입신호 없음 |
| 비금속 | 관찰 | 없음 | -2.00 | 0.16 | 3.00 | False | True | 업체수 감소 신호 · 생산유지·고용감소 신호 | 고용 축 진입신호 없음 |
| 섬유의복 | 관찰 | P | 0.00 | 0.03 | — | False | True | 가동 위축 신호 · 실질생산 위축 신호(PPI 조정) | 고용 축 진입신호 없음 (생산 12.3% 단독 감소 — 확인질문에 반영) |
| 기타 | 관찰 | 없음 | 25.00 | 0.15 | 2.00 | False | True | 업체수 감소 신호 | 고용 축 진입신호 없음 |
| 석유화학 | 관찰 | P | 54.00 | 0.54 | 1.00 | False | True | 가동 위축 신호 | 고용 축 진입신호 없음 (생산 6.7% 단독 감소 — 확인질문에 반영) |

### 관찰 결과

- 최신 2026Q2에는 기계·목재종이가 우선점검이고 나머지 8개 업종은 관찰이다. 추가확인은 0건이다.
- 기계는 고용 감소 규모가 크고(−3,979명, YoY −6.34%) A 상위경계와 생산 보강 신호를 함께 충족한다. 목재종이는 비율·상대열위·반복·생산 보강 신호가 함께 나타나지만 절대 감소 인원은 작다(−48명, YoY −10.08%).

### 해석

- 두 사례는 같은 우선점검이라도 Q2 규모와 확인 질문이 다르므로 진단카드로 나누어 인계한다.

> 주의: 관찰 업종도 정기 모니터링 대상이며, 생산 단독 감소나 외부 맥락신호는 확인질문에 남는다.

## 3단계. 중요 사례 진단카드

다기준 재검토는 해당 사례의 Triage 단계가 추가확인일 때만 표시한다. 최신 우선점검 2건은 ELECTRE/SMAA를 거치지 않고 외부근거·담당자 검토로 바로 이동한다.

### 왜 이걸 보나

카드는 CORE 진단 → Triage → (추가확인일 때만) 다기준 재검토 → 외부근거 → 해석 → 담당자 확인질문 → 인계 가능한 기존기능 순서로, explanation_trace.csv에 이미 계산된 값을 그대로 옮긴다. 업종마다 반복되는 형식이므로 표 대신 카드로 보여준다.

### 카드 해석 규칙 — 데이터로 확인한 것과 아직 추정인 것

각 카드의 "5. 해석"은 두 줄로 고정한다. "데이터로 확인"은 Triage가 그 업종을 선택한 신호·규모·지속 근거이고, "아직 추정"은 수주·자동화·외주화·휴업·투자·이직처럼 집계자료만으로는 확정할 수 없는 기업별 원인이다.

이 구분을 지우거나 두 줄을 하나로 합치지 않는다.

In [4]:
stage_order={'우선점검':0,'추가확인':1,'관찰':2}
for r in summary.assign(_o=summary.stage.map(stage_order)).sort_values('_o').itertuples():
    if r.stage not in ['우선점검','추가확인']: continue
    electre='해당 없음 — 우선점검은 외부근거·담당자 검토로 직접 이동'
    if r.stage=='추가확인':
        rr=review.loc[(review.industry==r.industry)&(review.quarter==r.quarter)]
        electre='결과 없음' if rr.empty else f"{rr.iloc[0].electre_stage}; 가능단계 {rr.iloc[0].possible_stages}; 민감={rr.iloc[0].smaa_parameter_sensitive}"
    display(Markdown(f"""### {r.industry}

#### 1. CORE 진단

- Q1: {r.q1_state_label}
- Q2: 고용 {r.q2_employment_delta:,.0f}명, YoY {r.q2_employment_yoy:.2f}%, 고용비중 {r.q2_employment_share:.2f}%
- Q3: 현재 상태 {r.q3_state_run_length:.0f}분기, {r.q3_transition}, 반복신호={r.q3_repeated_signal}

#### 2. Triage

- 단계: **{r.stage}**
- 선택 이유: {r.stage_reason}

#### 3. 다기준 재검토

- {electre}

#### 4. 외부근거

- {r.signal_profile}
- 범위 주의: 산단·창원시·경남·전국 자료가 혼재하므로 판정을 덮어쓰지 않음

#### 5. 해석

- 데이터로 확인: {r.stage_reason}
- 아직 추정: 기업별 수주·자동화·외주화·휴업·투자·이직 원인

#### 6. 담당자 확인 질문

- {r.check_question}
- {r.check_questions_context}

#### 7. 인계 가능한 기존 기능

- {r.first_owner}
- {r.handoff_review_functions}
"""))

### 기계

#### 1. CORE 진단

- Q1: 생산↓·고용↓
- Q2: 고용 -3,979명, YoY -6.34%, 고용비중 51.19%
- Q3: 현재 상태 2분기, S4 → S4, 반복신호=False

#### 2. Triage

- 단계: **우선점검**
- 선택 이유: 고용감소 6.3%(진입경계 5% 통과) · 산단 제조업 고용의 3.47% 감소(상위경계 2.0% 통과) → 보강: 생산 19.5% 감소

#### 3. 다기준 재검토

- 해당 없음 — 우선점검은 외부근거·담당자 검토로 직접 이동

#### 4. 외부근거

- 가동 위축 신호 · 실질생산 위축 신호(PPI 조정)
- 범위 주의: 산단·창원시·경남·전국 자료가 혼재하므로 판정을 덮어쓰지 않음

#### 5. 해석

- 데이터로 확인: 고용감소 6.3%(진입경계 5% 통과) · 산단 제조업 고용의 3.47% 감소(상위경계 2.0% 통과) → 보강: 생산 19.5% 감소
- 아직 추정: 기업별 수주·자동화·외주화·휴업·투자·이직 원인

#### 6. 담당자 확인 질문

- 수주잔량·가동률·휴업/감산·기업 수 변동·고용조정 계획 신고 여부를 확인한다(원인 후보이며 입증된 원인이 아님).
- 가동률 하락이 일시적 생산조정인지 수주·수요 감소에 따른 것인지 확인할 필요가 있다.
명목생산 변화에 가격효과가 포함되어 있으므로 실제 물량이 줄었는지 추가로 확인할 필요가 있다.
확인된 품목 기준 수출이 함께 감소하고 있어, 외부수요 약화 가능성을 보려면 수출 감소가 특정 품목·기업에 집중되는지 추가 확인할 필요가 있다.
창원시 제조업 전체에서 이직자 증가가 함께 관측되므로, 특정 사업장 조정 또는 노동이동과 관련된 것인지 추가 확인할 필요가 있다.
가동률과 전력사용의 방향이 달라 업종 구성·설비 특성·자료범위(산단 vs 시 전체) 차이를 추가 확인할 필요가 있다.
경남 제조업 업황 심리지수가 기준선(100)을 밑돌아, 업종 고유 요인인지 지역 전반의 경기 흐름인지 구분해 확인할 필요가 있다.
같은 업종의 전국 업황 심리지수도 기준선을 밑돌아, 관측된 변화가 산단 고유 현상인지 전국 업종 흐름인지 확인할 필요가 있다.
이 분기는 월별 원자료가 공표되지 않아 분기 내 지속 여부를 확인할 수 없다. 분기말 단일시점 값에만 의존한 판단임을 감안할 필요가 있다.
연간보정 개정본을 사용한 구간이다. 이후 재개정 시 값이 달라질 수 있음을 확인할 필요가 있다.

#### 7. 인계 가능한 기존 기능

- 기업지원 기능 + 고용지원 기능
- 산업동향 · 기업지원 / 산업동향 / 고용지원


### 목재종이

#### 1. CORE 진단

- Q1: 생산↓·고용↓
- Q2: 고용 -48명, YoY -10.08%, 고용비중 0.37%
- Q3: 현재 상태 2분기, S4 → S4, 반복신호=True

#### 2. Triage

- 단계: **우선점검**
- 선택 이유: 고용감소 10.1%(상위경계 10% 통과) · 산단평균 대비 6.4%p 열위(진입경계 통과) → 보강: 생산 12.9% 감소 / 직전 분기에도 진입신호(지속)

#### 3. 다기준 재검토

- 해당 없음 — 우선점검은 외부근거·담당자 검토로 직접 이동

#### 4. 외부근거

- 실질생산 위축 신호(PPI 조정)
- 범위 주의: 산단·창원시·경남·전국 자료가 혼재하므로 판정을 덮어쓰지 않음

#### 5. 해석

- 데이터로 확인: 고용감소 10.1%(상위경계 10% 통과) · 산단평균 대비 6.4%p 열위(진입경계 통과) → 보강: 생산 12.9% 감소 / 직전 분기에도 진입신호(지속)
- 아직 추정: 기업별 수주·자동화·외주화·휴업·투자·이직 원인

#### 6. 담당자 확인 질문

- 수주잔량·가동률·휴업/감산·기업 수 변동·고용조정 계획 신고 여부를 확인한다(원인 후보이며 입증된 원인이 아님).
- 명목생산 변화에 가격효과가 포함되어 있으므로 실제 물량이 줄었는지 추가로 확인할 필요가 있다.
창원시 제조업 전체에서 이직자 증가가 함께 관측되므로, 특정 사업장 조정 또는 노동이동과 관련된 것인지 추가 확인할 필요가 있다.
경남 제조업 업황 심리지수가 기준선(100)을 밑돌아, 업종 고유 요인인지 지역 전반의 경기 흐름인지 구분해 확인할 필요가 있다.
같은 업종의 전국 업황 심리지수도 기준선을 밑돌아, 관측된 변화가 산단 고유 현상인지 전국 업종 흐름인지 확인할 필요가 있다.
이 분기는 월별 원자료가 공표되지 않아 분기 내 지속 여부를 확인할 수 없다. 분기말 단일시점 값에만 의존한 판단임을 감안할 필요가 있다.
연간보정 개정본을 사용한 구간이다. 이후 재개정 시 값이 달라질 수 있음을 확인할 필요가 있다.

#### 7. 인계 가능한 기존 기능

- 기업지원 기능 + 고용지원 기능
- 산업동향 / 고용지원


## 4단계. 선택적 ELECTRE/SMAA와 Human-in-the-Loop

### 왜 이걸 보나

경로별로 ELECTRE 진입 여부와 최종 판단 주체가 규칙대로 지켜지는지, 추가확인 35건과 선택적 재검토 35건이 실제로 같은 집합인지 확인한다.

In [5]:
path_check=pd.DataFrame({'경로':['관찰','추가확인','우선점검'],'다음 단계':['정기 모니터링','선택적 ELECTRE/SMAA → 외부근거','외부근거'],'ELECTRE 필수':[False,True,False],'자동 stage 변경':[False,False,False],'최종 판단':['담당자','담당자','담당자']})
show_table(path_check)
print('전체 추가확인:',int(triage.stage.eq('추가확인').sum()),'| 선택 재검토:',len(review),'| ELECTRE stage 분포:',review.electre_stage.value_counts().to_dict(),'| overwrite:',int((review.triage_stage_preserved!='추가확인').sum()))

| 경로 | 다음 단계 | ELECTRE 필수 | 자동 stage 변경 | 최종 판단 |
| --- | --- | --- | --- | --- |
| 관찰 | 정기 모니터링 | False | False | 담당자 |
| 추가확인 | 선택적 ELECTRE/SMAA → 외부근거 | True | False | 담당자 |
| 우선점검 | 외부근거 | False | False | 담당자 |

전체 추가확인: 35 | 선택 재검토: 35 | ELECTRE stage 분포: {'PRIORITY': 20, 'CHECK': 9, 'UNDETERMINED': 4, 'OBSERVE': 2} | overwrite: 0


### 관찰 결과

- 전체 기간 추가확인은 35건이고 선택적 재검토도 35건으로 같다. Triage stage가 추가확인에서 다른 값으로 바뀐 사례(overwrite)는 0건이다.
- ELECTRE stage 분포는 PRIORITY 20, CHECK 9, UNDETERMINED 4, OBSERVE 2다.
- 관찰·추가확인·우선점검 세 경로 모두 최종 판단은 담당자이고, 어느 경로도 stage를 자동으로 바꾸지 않는다.

### Decision Box — 왜 담당자 최종판단(Human-in-the-Loop)을 남기는가

- <b>문제:</b> Triage와 선택적 ELECTRE/SMAA로 확인 순서와 다기준 재검토 결과까지 나온 뒤, 이 결과를 그대로 자동 확정으로 쓸 것인가.
- <b>선택:</b> 쓰지 않는다. 모든 경로의 마지막 단계는 담당자의 Human-in-the-Loop 확인이며, `human_decision_required`는 최신 10건 전부 True다.
- <b>근거:</b> ELECTRE stage는 UNDETERMINED가 4건 남아 있고 PRIORITY 20건도 SMAA 파라미터 민감도가 표시된 사례를 포함한다. 외부근거는 KEPCO 비식별 40.47%, 고용24 2026Q3 단면처럼 범위·시점이 CORE와 다르다. 집계자료 어느 것도 기업별 수주·자동화·외주화·휴업·투자·이직 원인을 확정하지 못한다.
- <b>결론에 미친 영향:</b> 06은 정책대상 자동 선정이나 사업 처방을 만들지 않는다. 진단카드는 확인질문과 인계 가능한 기존기능까지만 제시하고, 최종 판단과 기록은 담당자가 남긴다.

## 5단계. 기존 기능으로의 인계

아래는 자동 사업추천이 아니라 담당자가 현장 확인 후 검토할 수 있는 기존 기능의 지도다. 실제 기관명·사업 운영 여부·지원요건은 인계 시점에 다시 확인한다.

In [6]:
show_table(routing[['role','institution','unit','function','source_url']])

| role | institution | unit | function | source_url |
| --- | --- | --- | --- | --- |
| 산업·경제동향 | 창원상공회의소 | 지역·산단·업종 | 정기 경제동향 | https://changwoncci.korcham.net/front/board/boardContentsView.do?boardId=11175&contId=130203&menuId=3992 |
| 산업·고용동향 | 창원산업진흥원 | 지역·산업 | 산업·경제동향 및 고용동향 자료 | https://www.cwip.or.kr/ |
| 지역 위기진단 | 경남TP 위기지원센터 | 중소기업 밀집지역·기업 | 모니터링·심층현장조사·맞춤지원 | https://www.gntp.or.kr/introduce/staff |
| 기업 진단·고용지원 | 창원고용복지+센터 | 기업·구직자 | 기업 도약보장 패키지 등 진단·연계(운영 공지 시점 확인 필요) | https://www.moel.go.kr/local/changwon/news/reportexplan/view.do?bbs_seq=20240901144 |
| 기업 현장지원 | 창원산업진흥원 기업지원 기능·마이스터센터 | 기업 | 현장애로 컨설팅·기술지원 | https://www.cwip.or.kr/ |
| 인력·훈련 연계 | 경남지역인적자원개발위원회 | 산업·기업 인력수요 | 수요조사·맞춤형 훈련 연계 | https://gnhrd.or.kr/ |

### 관찰 결과

- 인계 지도는 산업·경제동향, 산업·고용동향, 지역 위기진단, 기업 진단·고용지원, 기업 현장지원, 인력·훈련 연계까지 6개 기능으로 구성된다.
- 기계·목재종이 카드의 "인계 가능한 기존 기능"은 이 지도 중 산업동향·기업지원·고용지원 항목을 가리킨다.

> 주의: 운영 공지 시점은 기관마다 다르므로 인계 시점에 다시 확인한다.

## 6단계. 담당자 기록 필드와 다음 분기 재점검

1. 기업·현장 확인 대상과 근거
2. 수주·가동·설비·외주·인력·고용조정 관련 확인 사실
3. 외부자료와 CORE의 일치/불일치 이유
4. 기존 지원기능 검토 결과(연계/미연계/추가자료 필요)
5. 다음 분기 재점검 시 비교할 지표

이 기록은 모형의 자동 학습 label로 즉시 쓰지 않는다. 정의·수집방식·검증절차가 합의된 뒤 별도 평가자료로 관리한다.

# 정리

## 확인한 것

- 최신 2026Q2는 기계·목재종이가 우선점검이며 추가확인은 0건이다. 전체 기간 180건은 관찰 128·추가확인 35·우선점검 17이며, 추가확인 35건은 선택적 재검토 35건과 정확히 일치하고 overwrite는 0건이다.
- 모든 경로의 마지막 단계는 담당자 확인이며, 최신 10건 모두 human_decision_required = True다.
- 인계 지도는 6개 기능으로 구성되며, 두 우선점검 업종의 진단카드는 이 지도 중 해당 기능을 가리킨다.

## 말할 수 없는 것

- 이 결과는 위기 확정, 기업별 원인, 지원대상 선정, 사업 처방, 예산 배분 또는 정책효과를 자동으로 결정하지 않는다.
- 담당자 기록은 아직 남지 않았으므로 이번 분기의 실제 현장 확인 결과는 이 노트북만으로 알 수 없다.

## 다음 단계

담당자 확인 결과를 기록하고 다음 분기에 재점검한다.